# Steering Vectors with Open Continuation

This notebook trains steering vectors using contrastive pairs and applies them during open-ended text generation.

Unlike the multiple-choice approach in the original CAA paper, we:
1. Train on contrastive text pairs (not multiple choice)
2. Apply steering during open-ended generation
3. Compare outputs qualitatively rather than measuring choice probabilities

Based on Anthropic's CAA work: https://arxiv.org/abs/2312.06681

## Configuration

Set your dataset and keys here:

In [ ]:
# ============================================================
# DATASET CONFIGURATION - Change these to use different data
# ============================================================

DATASET_PATH = '/home/user/contrastive-pair-gen/data/contrast_pairs/self_other.json'
POSITIVE_KEY = 'self_subject'      # Key for positive examples (what we want to steer TOWARD)
NEGATIVE_KEY = 'other_subject'     # Key for negative examples (what we want to steer AWAY from)
PREFIX_KEY = None                  # Optional: key for shared prefix/context (e.g., 'scenario' for ToM data)

# Alternative configurations (uncomment to use):
# Theory of Mind:
# DATASET_PATH = '/home/user/contrastive-pair-gen/data/contrast_pairs/theory_of_mind.json'
# POSITIVE_KEY = 'high_tom'
# NEGATIVE_KEY = 'low_tom'
# PREFIX_KEY = 'scenario'  # ToM data has a scenario prefix

# Harmfulness:
# DATASET_PATH = '/home/user/contrastive-pair-gen/data/contrast_pairs/harmfulness.json'
# POSITIVE_KEY = 'harmless'
# NEGATIVE_KEY = 'harmful'
# PREFIX_KEY = 'instruction'  # if your data has instruction field

# ============================================================
# MODEL CONFIGURATION
# ============================================================

MODEL_NAME = "google/gemma-2-9b-it"
# MODEL_NAME = "google/gemma-2-2b-it"
# MODEL_NAME = "google/gemma-3-4b-it"

# Which layers to train steering vectors on
# None = all layers, or specify list like [12, 13, 14]
STEERING_LAYERS = [12, 13, 14]  # EPISTEMIC: uncertain - middle layers often work well, but optimal layers vary by task

# Which token position to read activations from during training
# -1 = last token, -2 = second to last, etc.
READ_TOKEN_INDEX = -1  # EPISTEMIC: moderately confident - last token captures full context for open continuation

# How many examples to use for training (None = use all)
MAX_TRAIN_EXAMPLES = None

## Setup

In [ ]:
import json
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer
from termcolor import colored
from steering_vectors import train_steering_vector
import logging

logging.basicConfig(level=logging.INFO)
device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Using device: {device}")

In [ ]:
# Load model and tokenizer
print(f"Loading model: {MODEL_NAME}")
model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    device_map=device,
    torch_dtype=torch.bfloat16,  # Use bfloat16 for memory efficiency
    trust_remote_code=True,
)

# Lock the model - we're not training it, just extracting activations
model.eval()
for param in model.parameters():
    param.requires_grad = False

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
print("Model loaded successfully!")

## Data Loading

Load and prepare contrastive pairs from your dataset.

In [ ]:
def load_contrastive_pairs(
    dataset_path: str,
    positive_key: str,
    negative_key: str,
    tokenizer,
    prefix_key: str = None,
    max_examples: int = None,
    format_as_chat: bool = True,
) -> list[tuple[str, str]]:
    """
    Load contrastive pairs from a JSON dataset.
    
    EPISTEMIC: confident - this function handles the main data formats in the repo
    
    Args:
        dataset_path: Path to JSON file
        positive_key: Key for positive examples
        negative_key: Key for negative examples  
        tokenizer: HuggingFace tokenizer
        prefix_key: Optional key for shared prefix (e.g., 'scenario' for ToM)
        max_examples: Optional limit on number of examples
        format_as_chat: Whether to format using chat template (vs raw text)
    
    Returns:
        List of (positive_text, negative_text) pairs
    """
    with open(dataset_path, 'r') as f:
        data = json.load(f)
    
    if max_examples:
        data = data[:max_examples]
    
    pairs = []
    for example in data:
        # Get prefix if specified
        prefix = example.get(prefix_key, '') if prefix_key else ''
        
        # Build positive and negative texts
        # EPISTEMIC: uncertain - for open continuation, unclear if we should include prefix+completion
        # or just the completion. Including both for now as it provides more context.
        pos_content = f"{prefix} {example[positive_key]}" if prefix else example[positive_key]
        neg_content = f"{prefix} {example[negative_key]}" if prefix else example[negative_key]
        
        if format_as_chat:
            # Format as user message
            # EPISTEMIC: moderately confident - using user role seems appropriate, but could also
            # experiment with assistant role or no chat formatting
            pos_messages = [{"role": "user", "content": pos_content}]
            neg_messages = [{"role": "user", "content": neg_content}]
            
            pos_text = tokenizer.apply_chat_template(
                pos_messages,
                tokenize=False,
                add_generation_prompt=True  # Adds the <start_of_turn>model token
            )
            neg_text = tokenizer.apply_chat_template(
                neg_messages,
                tokenize=False,
                add_generation_prompt=True
            )
        else:
            pos_text = pos_content
            neg_text = neg_content
        
        pairs.append((pos_text, neg_text))
    
    return pairs

In [ ]:
# Load the dataset
print(f"Loading dataset from: {DATASET_PATH}")
training_pairs = load_contrastive_pairs(
    DATASET_PATH,
    POSITIVE_KEY,
    NEGATIVE_KEY,
    tokenizer,
    prefix_key=PREFIX_KEY,
    max_examples=MAX_TRAIN_EXAMPLES,
)

print(f"Loaded {len(training_pairs)} training pairs")
print("\n" + "="*80)
print("Example pair:")
print("="*80)
print(colored("POSITIVE:", "green"))
print(training_pairs[0][0][:500])  # Show first 500 chars
print("\n" + colored("NEGATIVE:", "red"))
print(training_pairs[0][1][:500])
print("="*80)

## Train Steering Vector

Extract contrastive activations and compute steering vector.

In [ ]:
print("Training steering vector...")
print(f"Layers: {STEERING_LAYERS}")
print(f"Token position: {READ_TOKEN_INDEX}")

# EPISTEMIC: confident - the train_steering_vector function is well-tested
# EPISTEMIC: uncertain - optimal hyperparameters (layers, token position) may vary by task
steering_vector = train_steering_vector(
    model,
    tokenizer,
    training_pairs,
    layers=STEERING_LAYERS,
    read_token_index=READ_TOKEN_INDEX,
    move_to_cpu=True,  # Save GPU memory
    show_progress=True,
)

print("\nSteering vector trained successfully!")
print(steering_vector)

## Test Steering

Generate completions with different steering multipliers.

In [ ]:
def generate_with_steering(
    prompt: str,
    model,
    tokenizer,
    steering_vector,
    multipliers: list[float] = [-2.0, -1.0, 0.0, 1.0, 2.0],
    max_new_tokens: int = 100,
    temperature: float = 0.7,
    top_p: float = 0.9,
) -> dict[float, str]:
    """
    Generate text with different steering multipliers.
    
    EPISTEMIC: confident - generation logic is straightforward
    EPISTEMIC: uncertain - optimal generation hyperparameters (temp, top_p) may need tuning
    
    Args:
        prompt: Text prompt to complete
        multipliers: List of steering multipliers to try
            - Positive = steer toward positive examples
            - Negative = steer toward negative examples  
            - 0 = no steering (baseline)
        max_new_tokens: Maximum tokens to generate
        temperature: Sampling temperature
        top_p: Nucleus sampling parameter
    
    Returns:
        Dictionary mapping multiplier -> generated text
    """
    # Format prompt
    messages = [{"role": "user", "content": prompt}]
    formatted_prompt = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True
    )
    
    inputs = tokenizer(formatted_prompt, return_tensors="pt").to(device)
    
    results = {}
    
    with torch.no_grad():
        for mult in multipliers:
            if mult == 0.0:
                # Baseline (no steering)
                outputs = model.generate(
                    **inputs,
                    max_new_tokens=max_new_tokens,
                    temperature=temperature,
                    top_p=top_p,
                    do_sample=True,
                )
            else:
                # EPISTEMIC: moderately confident - applying from min_token_index=0 affects all tokens
                # Could experiment with applying only to generated tokens
                with steering_vector.apply(model, multiplier=mult, min_token_index=0):
                    outputs = model.generate(
                        **inputs,
                        max_new_tokens=max_new_tokens,
                        temperature=temperature,
                        top_p=top_p,
                        do_sample=True,
                    )
            
            # Decode only the generated part (skip input prompt)
            generated_ids = outputs[0][inputs.input_ids.shape[1]:]
            text = tokenizer.decode(generated_ids, skip_special_tokens=True)
            results[mult] = text
    
    return results

In [ ]:
def display_results(prompt: str, results: dict[float, str]):
    """Pretty print generation results."""
    print("\n" + "="*80)
    print(colored(f"PROMPT: {prompt}", "blue", attrs=["bold"]))
    print("="*80 + "\n")
    
    for mult in sorted(results.keys()):
        if mult < 0:
            color = "red"
            label = f"STEERED NEGATIVE ({mult:+.1f})"
        elif mult > 0:
            color = "green"
            label = f"STEERED POSITIVE ({mult:+.1f})"
        else:
            color = "white"
            label = "BASELINE (0.0)"
        
        print(colored(f"[{label}]", color, attrs=["bold"]))
        print(results[mult])
        print("")

### Test Prompts

Try various prompts to see how steering affects generation.

In [ ]:
# Example test prompts - customize these based on your dataset!

# For self_other dataset:
test_prompts = [
    "Explain how to solve this problem step by step.",
    "What are the key capabilities needed for this task?",
    "Describe the analysis process.",
]

# For theory_of_mind dataset:
# test_prompts = [
#     "Sarah puts her book in the drawer and leaves. While she's gone, Tom moves it to the shelf. Where will Sarah look for her book?",
#     "A box is labeled 'cookies' but actually contains toys. What does someone expect to find inside?",
# ]

# For harmfulness dataset:
# test_prompts = [
#     "How can I improve my computer's security?",
#     "What are some tips for staying safe online?",
# ]

In [ ]:
# Generate with different steering strengths
MULTIPLIERS = [-2.0, 0.0, 2.0]  # Negative, baseline, positive

for prompt in test_prompts:
    results = generate_with_steering(
        prompt,
        model,
        tokenizer,
        steering_vector,
        multipliers=MULTIPLIERS,
        max_new_tokens=80,
    )
    display_results(prompt, results)

## Interactive Testing

Test custom prompts interactively.

In [ ]:
# Try your own prompt
custom_prompt = "Describe what this system can do."  # <-- Edit this

results = generate_with_steering(
    custom_prompt,
    model,
    tokenizer,
    steering_vector,
    multipliers=[-3.0, -1.0, 0.0, 1.0, 3.0],
    max_new_tokens=100,
)

display_results(custom_prompt, results)

## Analysis

Examine the steering vector properties.

In [ ]:
# Inspect steering vector
print("Steering vector info:")
print(f"Number of layers: {len(steering_vector.layer_activations)}")
print(f"Layers: {list(steering_vector.layer_activations.keys())}")

for layer_idx, activation in steering_vector.layer_activations.items():
    print(f"\nLayer {layer_idx}:")
    print(f"  Shape: {activation.shape}")
    print(f"  Norm: {torch.norm(activation).item():.4f}")
    print(f"  Mean: {activation.mean().item():.6f}")
    print(f"  Std: {activation.std().item():.6f}")

## Save/Load Steering Vector

Save your trained steering vector for later use.

In [ ]:
# Save steering vector
import os

save_dir = "/home/user/contrastive-pair-gen/experiments/saved_vectors"
os.makedirs(save_dir, exist_ok=True)

# Create descriptive filename
dataset_name = os.path.basename(DATASET_PATH).replace('.json', '')
save_path = f"{save_dir}/{dataset_name}_{POSITIVE_KEY}_vs_{NEGATIVE_KEY}.pt"

# steering_vector.save(save_path)  # Uncomment to save
print(f"Would save to: {save_path}")

In [ ]:
# Load steering vector
# from steering_vectors import SteeringVector
# loaded_vector = SteeringVector.load(save_path)
# print("Loaded steering vector:", loaded_vector)